
### Data Ingestion Strategy & Unity Catalog Architecture

**File:** `notebooks/01_data_ingestion.py`

This notebook serves as the foundational data pipeline for the `mcp-fra-safety` server. It extracts the last 10 years (2016–2026) of Federal Railroad Administration (FRA) safety data from `data.transportation.gov` and stages it in Databricks Unity Catalog as high-performance Delta Tables.

**Key Technical Implementations:**

* **Resilient API Pagination (Criterion 2 - Reliability):** The default Socrata API frequently times out on large queries or enforces a 1,000-row limit. This ingestion script implements a robust paginated chunking mechanism using `$limit=50000` and `$offset` parameters, ensuring zero data loss and uninterrupted downloads across millions of rows.
* **Delta Table Sanitation (Criterion 1 - Maintainability):** Raw FRA CSVs contain spaces, dashes, and parentheses in their headers. The script dynamically applies `sanitize_column_names()` to strip illegal characters, ensuring strict compatibility with the Databricks SQL Warehouse backend.
* **Query Optimization (Criterion 2 - Performance):** After saving the raw data into Unity Catalog (`users.neeraj_koul.fra_raw`), the script runs `OPTIMIZE` and `ANALYZE TABLE COMPUTE STATISTICS`. This optimizes data file layouts (Z-ordering) and caches statistics, allowing the MCP tools to execute complex cross-dataset SQL joins in milliseconds.

**Datasets Ingested:**

1. `form_54_accidents` (Rail Equipment Accidents/Incidents)
2. `form_55a_casualties` (Injury/Illness Continuation)
3. `form_55_operational` (Operational Data: miles, hours, passengers)
4. `form_57_crossings` (Highway-Rail Grade Crossing Incidents)

**Usage:**
Run this notebook once on a standard Databricks compute cluster to populate the Unity Catalog environment before deploying the MCP server application.   This can be re-run if update is required ( that should be for prod environment)


In [0]:
import requests
from pyspark.sql.functions import col

CATALOG = "users"
SCHEMA = "neeraj_koul"
VOLUME = "fra_raw"
START_YEAR = 2016
PAGE_SIZE = 50000

# 1. Create schema and volume in Unity Catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

# FRA Socrata Endpoints mapped to their respective year columns for PySpark filtering
endpoints = {
    "form_54_accidents": {
        "url": "https://data.transportation.gov/resource/85tf-25kj.csv",
        "year_col": "year4"
    },
    "form_55a_casualties": {
        "url": "https://data.transportation.gov/resource/rash-pd2d.csv",
        "year_col": "iyr"
    },
    "form_55_operational": {
        "url": "https://data.transportation.gov/resource/m8i6-zdsy.csv",
        "year_col": "year"
    },
    "form_57_crossings": {
        "url": "https://data.transportation.gov/resource/7wn6-i5b9.csv",
        "year_col": "iyr"
    }
}

def sanitize_column_names(df):
    """Sanitize column names so they are compatible with Unity Catalog Delta Tables."""
    for c in df.columns:
        new_c = c.strip().lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
        df = df.withColumnRenamed(c, new_c)
    return df

# 2. Download iteratively to avoid Socrata timeouts
for table_name, config in endpoints.items():
    base_url = config["url"]
    year_col = config["year_col"]
    csv_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{table_name}_raw.csv"
    offset = 0
    first_page = True

    print(f"Downloading {table_name}...")
    
    with open(csv_path, "wb") as f:
        while True:
            params = {
                "$order": ":id", # Required for reliable pagination
                "$limit": PAGE_SIZE,
                "$offset": offset,
            }
            resp = requests.get(base_url, params=params, timeout=120)
            resp.raise_for_status()

            lines = resp.content.splitlines(keepends=True)
            if len(lines) <= 1:
                break # We've hit the end of the dataset

            if first_page:
                f.writelines(lines)
                first_page = False
            else:
                f.writelines(lines[1:]) # Skip headers for subsequent pages

            offset += PAGE_SIZE

    # 3. Read into Spark, sanitize, filter for 2016-2026, and write to Delta
    print(f"Processing {table_name} into Delta Table...")
    
    df = (
        spark.read.option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(csv_path)
    )
    
    # Clean columns for Delta compatibility
    df = sanitize_column_names(df)
    
    # Filter for the last 10 years based on the specific form's year column
    if year_col in df.columns:
        df = df.filter(col(year_col) >= START_YEAR)
    
    # Write to Unity Catalog
    table_path = f"{CATALOG}.{SCHEMA}.{table_name}"
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(table_path)
    
    # 4. Optimize for Databricks SQL Warehouse fast queries
    spark.sql(f"OPTIMIZE {table_path}")
    spark.sql(f"ANALYZE TABLE {table_path} COMPUTE STATISTICS")
    
    print(f"✅ Saved {table_path}: {df.count():,} rows (2016+)\n")

print("All datasets successfully ingested and optimized!")